# Phase 1 - Plant disease detection model (MobileNetV2 on PlantVillage)

Capstone Project 7, Group 4. This notebook trains, evaluates and publishes the disease classifier
described in the proposal (Section 9, Phase 1) and hands it over to Phase 5.

**What it does**

1. Downloads PlantVillage (color) from Kaggle via `kagglehub`.
2. Fine-tunes `google/mobilenet_v2_1.0_224` with augmentation for tilted / inverted / blurry photos.
3. Reports accuracy, per-class precision / recall / F1, confusion matrix, top-3 and a confidence-threshold sweep.
4. Tests on the labelled real-world set through the live-app chain (target: at least 50% end-to-end)
   and republishes the Hub model card whenever the set, the weights or the threshold change.
5. Pushes the model, reports and model card to **https://huggingface.co/neghani/plantvillage-mobilenetv2** and tags `v1.0`.
6. Optional stage 2 (section 7b): continues from `v1.0` on PlantVillage replay plus PlantDoc field-photo
   crops (cloned and prepared by `model/prepare_plantdoc.py`), tested on held-out field photos; nothing is published automatically.

**Only re-testing a new or changed real-world set?** Run sections 1, 2 and 7. No training needed:
section 7 falls back to the published `v1.0` weights.

All logic lives in `model/*.py` in the repository; this notebook only runs those scripts.

**How to test (local vs Colab)**

- **Local (CPU):** edit `model/*.py`, smoke-test tomato / 40 images / 1 epoch, then `git push`.
- **Colab (GPU):** do **not** `files.upload()` code. Re-run the sync cell below. It always
  replaces tracked files with `origin/main` (`notebooks/colab_sync.py`) so you never re-test a stale upload.
- **Real-world images:** git-ignored. Keep a zip on Drive and pass `--real-world-from` (always replaces).

**Before you start (Colab)**

- Runtime -> Change runtime type -> **T4 GPU** (or better).
- Colab secrets (key icon in the left bar): add `HF_TOKEN` (a *write* token from https://huggingface.co/settings/tokens).
  Optional: `KAGGLE_USERNAME` and `KAGGLE_KEY` if `kagglehub` asks for them (the dataset is public, so usually not needed).

In [ ]:
!nvidia-smi -L || echo "No GPU - training will be very slow; switch the runtime to a GPU."

## 1. Sync the repository (always replace from GitHub)

Fetches `notebooks/colab_sync.py` from GitHub, then hard-resets tracked files to `origin/main`.
Checkpoints in `model/outputs/` and images under `model/data/real_world/` are left alone.

After you push a local fix, **re-run this cell** (or the one-file cell after it). Do not upload `.py` files.

In [ ]:
import os, urllib.request
from pathlib import Path

REPO = "neghani/iiith-group4-capstone-plat-leaf-decease-detection"
RAW = f"https://raw.githubusercontent.com/{REPO}/main/notebooks/colab_sync.py"
helper = Path("/tmp/colab_sync.py")
try:
    urllib.request.urlretrieve(RAW, helper)
    print("Fetched", RAW)
except Exception as exc:
    local = Path("notebooks/colab_sync.py")
    if not local.exists():
        raise RuntimeError("Could not fetch colab_sync.py from GitHub; clone the repo first.") from exc
    helper = local
    print("Using local", helper, "because raw fetch failed:", exc)
%run $helper
os.chdir(os.environ.get("REPO_ROOT", os.getcwd()))
print("cwd", os.getcwd())

Optional: replace **one** file after a local push (still from GitHub, always overwrite). Change the path and re-run. Do not `files.upload()`.

In [ ]:
# Re-run after `git push`. Always overwrites the listed file(s) from origin/main.
# Uncomment extra --file flags as needed.
!python notebooks/colab_sync.py --skip-install --file model/train.py
# !python notebooks/colab_sync.py --skip-install --file model/evaluate.py --file model/predict.py --file model/common.py

## 2. Log in to Hugging Face

The push target is fixed to `neghani/plantvillage-mobilenetv2`. If you run this with a different account, set `HF_MODEL_ID` to your own namespace below.

In [ ]:
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN to Colab secrets (enable 'Notebook access') or export HF_TOKEN."
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "neghani/plantvillage-mobilenetv2")
me = HfApi().whoami()["name"]
expected_owner = HF_MODEL_ID.split("/")[0]
assert me == expected_owner or expected_owner in [o["name"] for o in HfApi().whoami().get("orgs", [])], (
    f"Logged in as '{me}' but HF_MODEL_ID is '{HF_MODEL_ID}'. "
    f"Either log in as '{expected_owner}' or set os.environ['HF_MODEL_ID'] = '{me}/plantvillage-mobilenetv2' before this cell."
)
os.environ["HF_MODEL_ID"] = HF_MODEL_ID
print("Logged in as", me, "-> will push to", HF_MODEL_ID)

## 3. Download PlantVillage (color)

In [ ]:
import kagglehub, os
root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Downloaded to", root)

import sys
sys.path.insert(0, "model")
from common import resolve_data_dir
DATA_DIR = str(resolve_data_dir(root))
os.environ["DATA_DIR"] = DATA_DIR
classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print(len(classes), "classes, e.g.", classes[:5])
print("images:", sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes))

## 4. Quick smoke test (optional, ~2 min)

Tomato classes only, 60 images per class, 1 epoch. Confirms the pipeline before spending GPU time
on the full run. Nothing is pushed.

In [ ]:
!python model/train.py \
  --data_dir "$DATA_DIR" --crop_filter Tomato --max_samples_per_class 60 \
  --output_dir model/outputs/smoke \
  --do_train --do_eval --num_train_epochs 1 \
  --per_device_train_batch_size 32 --per_device_eval_batch_size 64 \
  --learning_rate 5e-4 --fp16 --report_to none --save_strategy no --logging_steps 5
!python model/evaluate.py --model model/outputs/smoke

## 5. Full training run

All 38 classes (~54k images). On a T4 one epoch is roughly 8-10 minutes with the settings below;
5 epochs is a good first full run. The best checkpoint by validation macro-F1 is kept, the test
split is scored, and everything is pushed to the Hub as soon as training finishes
(`--push_to_hub`), so a Colab disconnect after this cell does not lose the model.

To try the ResNet fallback from the proposal, change `--model_name_or_path microsoft/resnet-50`
and `--output_dir model/outputs/plantvillage-resnet50` (and `--hub_model_id` if you want a
separate repo).

In [ ]:
import os
OUTPUT_DIR = "model/outputs/plantvillage-mobilenetv2"
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
!python model/train.py \
  --model_name_or_path google/mobilenet_v2_1.0_224 \
  --data_dir "$DATA_DIR" \
  --output_dir "$OUTPUT_DIR" \
  --do_train --do_eval \
  --num_train_epochs 5 \
  --per_device_train_batch_size 64 --per_device_eval_batch_size 128 \
  --learning_rate 5e-4 --weight_decay 0.01 --warmup_steps 0.05 --lr_scheduler_type cosine \
  --fp16 --dataloader_num_workers 2 \
  --eval_strategy epoch --save_strategy epoch --save_total_limit 2 \
  --load_best_model_at_end --metric_for_best_model macro_f1 --greater_is_better True \
  --logging_steps 50 --report_to none --seed 42 \
  --push_to_hub --hub_model_id "$HF_MODEL_ID" --hub_strategy end

## 6. Evaluate on the held-out test split and build the model card

Produces `reports/` (metrics.json, classification report, confusion matrix, threshold sweep) and fills `README.md`; `--push` uploads both to the Hub.

In [ ]:
!python model/evaluate.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --push

In [ ]:
from IPython.display import Image, display
import json, os
display(Image(filename=os.path.join(os.environ["OUTPUT_DIR"], "reports", "confusion_matrix.png"), width=900))
m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy", "recommended_confidence_threshold")})

## 7. Real-world test set (target: at least 50% end-to-end) - publishes on change

The labelled set lives in `model/data/real_world_set/` (`labels.csv` with a `tune` / `final` split,
`sources.csv`, optional `card_notes.md`, and `images/`). The GitHub repo is public, so the images are
git-ignored and come from Drive: `MyDrive/plant-disease/real_world_set.zip` (zip the whole folder).

**New or changed set:** replace that zip on Drive and re-run this section. The next cell wipes the
local set and copies the zip in. The eval cell then runs the live-app chain (API quality gate + leaf
crop -> model) on `tune` and `final`, and compares a fingerprint of `labels.csv` + every image, the
model weights and the threshold with `reports/real_world_set/<split>/run.json` on the Hub:

- something changed (or the card was overwritten) -> results are uploaded and the **Hub README's
  real-world section and `model-index` are regenerated** from the `final` split;
- nothing changed -> it prints `unchanged` and pushes nothing.

Set `FORCE_PUBLISH = True` to republish anyway.

In [ ]:
# Always replace the labelled set from Drive (change the path if yours differs).
# Zip layout: real_world_set/{labels.csv, sources.csv, card_notes.md, images/<Label>/*.jpg}
# (the folder may also be the zip root). Every file listed in labels.csv must be present.
REAL_WORLD_SET_SRC = "/content/drive/MyDrive/plant-disease/real_world_set.zip"
!python notebooks/colab_sync.py --skip-install --real-world-set-from $REAL_WORLD_SET_SRC

In [ ]:
import json, os
from pathlib import Path

FORCE_PUBLISH = False

# Model under test: this session's run if it exists, else the published release.
out_dir = Path(os.environ.get("OUTPUT_DIR", "model/outputs/plantvillage-mobilenetv2"))
if (out_dir / "model.safetensors").exists():
    EVAL_MODEL, EVAL_REV = str(out_dir), ""
else:
    EVAL_MODEL, EVAL_REV = os.environ["HF_MODEL_ID"], os.environ.get("HF_MODEL_REVISION", "v1.0")
# Crop strategy + threshold chosen on the tune split by model/eval_strategies.py (= api/app/config.py).
STRATEGY, THRESHOLD = "B", 0.85
print(f"Testing {EVAL_MODEL}{'@' + EVAL_REV if EVAL_REV else ''} with strategy {STRATEGY} at threshold {THRESHOLD}")

rev_flag = f"--revision {EVAL_REV}" if EVAL_REV else ""
change_flag = "" if FORCE_PUBLISH else "--only_if_changed"
# tune first (thresholds are chosen there), final last: only final rewrites the Hub README.
for split in ("tune", "final"):
    !python model/eval_real_world_set.py --set_dir model/data/real_world_set --split {split} \
      --predictor app_predictor:diagnose --strategy {STRATEGY} --threshold {THRESHOLD} \
      --model "{EVAL_MODEL}" {rev_flag} --hub_model_id "$HF_MODEL_ID" \
      --out model/outputs/real_world/{split} --push {change_flag}

In [ ]:
from IPython.display import Markdown, display
report = Path("model/outputs/real_world/final/report.md")
display(Markdown(report.read_text()) if report.exists() else Markdown("_No final report yet._"))
print(f"Hub card: https://huggingface.co/{os.environ['HF_MODEL_ID']}")

### Optional: ad-hoc photo folder (not published to the card)

Loose photos in `model/data/real_world/<ClassName>/` from `MyDrive/plant-disease/real_world.zip`,
scored full-frame by `model/real_world_eval.py`. Kept for quick checks; the card's real-world section
comes from the labelled set above.

In [ ]:
# REAL_WORLD_SRC = "/content/drive/MyDrive/plant-disease/real_world.zip"
# !python notebooks/colab_sync.py --skip-install --real-world-from $REAL_WORLD_SRC
# !python model/real_world_eval.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID"

## 7b. Stage 2: fine-tune on field photos (PlantDoc) and keep PlantVillage

PlantVillage is one leaf on a plain background, so `v1.0` scores about 12% end-to-end on field-style leaves
(section 7). This stage **continues from the published `v1.0` weights** on a mix of

- **PlantVillage replay** (400 images per class from the training split; its validation and test splits are
  the same as for `v1.0`, so the PlantVillage test score stays comparable), and
- **PlantDoc leaf crops**: web field photos with one box per leaf, cropped 256x256 exactly like the live
  app crops a leaf, mapped to the same 38 labels (`model/plantdoc_map.py`).

`model/prepare_plantdoc.py` clones PlantDoc if it is not there yet and keeps leakage out of training:
every photo behind `real_world_set` (and any near-duplicate of it) and PlantDoc's own TEST photos are
removed, and train / validation are split by photo group, not by crop. PlantDoc's TEST split becomes a
second held-out set (`model/data/plantdoc_test`, hundreds of crops instead of 50).

**Needs this session:** section 2 (HF login), section 3 (`DATA_DIR`) and section 7's copy of the
real-world set (its images are what gets excluded). **Nothing here pushes to the Hub or tags a release.**

**Read the numbers with care.** The training crops, `real_world_set` and `plantdoc_test` all come from the
same web-sourced PlantDoc collection, so gains on them overstate what a farmer's phone photo will get.
The honest test is the trainer's `full_photos` set once it exists.

In [ ]:
# Clones PlantDoc into model/data/plantdoc_src if needed (~700 MB), then writes the crops.
!python model/prepare_plantdoc.py --exclude_set model/data/real_world_set

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown(Path("model/data/plantdoc_train/prep_report.md").read_text()))

### Train from `v1.0` (about 25-35 min on a T4)

Learning rate is 1e-4 (a continuation, not a fresh run). The best checkpoint is chosen by **PlantDoc
validation** macro-F1; PlantVillage validation is logged next to it (`eval_plantvillage_*`) so forgetting is visible.
Change the numbers here only based on the validation metrics, never on `real_world_set` final.

In [ ]:
import os
STAGE2_DIR = "model/outputs/plantvillage-plantdoc-mobilenetv2"
os.environ["STAGE2_DIR"] = STAGE2_DIR
!python model/train.py \
  --model_name_or_path "$HF_MODEL_ID" --model_revision v1.0 \
  --data_dir "$DATA_DIR" \
  --extra_data_dir model/data/plantdoc_train --extra_repeat 3 --replay_per_class 400 \
  --output_dir "$STAGE2_DIR" \
  --do_train --do_eval \
  --num_train_epochs 6 \
  --per_device_train_batch_size 64 --per_device_eval_batch_size 128 \
  --learning_rate 1e-4 --weight_decay 0.01 --warmup_steps 0.05 --lr_scheduler_type cosine \
  --fp16 --dataloader_num_workers 2 \
  --eval_strategy epoch --save_strategy epoch --save_total_limit 2 \
  --load_best_model_at_end --metric_for_best_model plantdoc_macro_f1 --greater_is_better True \
  --logging_steps 50 --report_to none --seed 42

### Did PlantVillage get worse?

Same held-out PlantVillage test split as `v1.0` (99.12% top-1, 98.58% macro-F1). A drop of more than about
1 point means too little replay or too high a learning rate: raise `--replay_per_class`, lower the rate, retrain.

In [ ]:
!python model/evaluate.py --model "$STAGE2_DIR" --reports_dir "$STAGE2_DIR/reports"
import json
m = json.load(open(os.path.join(STAGE2_DIR, "reports", "metrics.json")))
print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy")}, "(v1.0: 0.9912, 0.9858, 0.9994)")

### Threshold on the real-world **tune** split, then **final** once

Same rule as `eval_strategies.py` (strategy B; the threshold with the best pass rate among those where at
least 70% of "ok" answers are correct). Results go to `model/outputs/stage2/`, so the `v1.0` report and its
one-shot final lock are untouched. Final runs once for these weights.

In [ ]:
!python model/eval_strategies.py --split tune --strategy B --model "$STAGE2_DIR" --out model/outputs/stage2
import json
sel = json.load(open("model/outputs/stage2/selection.json"))
THR = sel["threshold"]
print("chosen", sel["strategy"], "@", THR, "-", sel["reason"])

In [ ]:
# Run once. A second run for the same weights is refused.
!python model/eval_strategies.py --split final --strategy B --threshold {THR} --model "$STAGE2_DIR" --out model/outputs/stage2

### PlantDoc held-out TEST (never trained on) - new weights vs `v1.0`

Each model at the threshold chosen for it on the tune split (`v1.0`: 0.85).

In [ ]:
!python model/eval_real_world_set.py --set_dir model/data/plantdoc_test --split all \
  --predictor app_predictor:diagnose --strategy B --threshold {THR} --model "$STAGE2_DIR" \
  --out model/outputs/stage2/plantdoc_test
!python model/eval_real_world_set.py --set_dir model/data/plantdoc_test --split all \
  --predictor app_predictor:diagnose --strategy B --threshold 0.85 --model "$HF_MODEL_ID" --revision v1.0 \
  --out model/outputs/stage2/plantdoc_test_v1

### If it is better: publish as `v1.1` (optional, manual)

Only do this when the PlantDoc TEST and real-world **final** numbers improve and PlantVillage stays within
about 1 point. `publish.py` uploads to the repo and adds the tag `v1.1`; the `v1.0` tag keeps pointing at the
old weights, so the API and Space stay on `v1.0` until you change `HF_MODEL_REVISION` (`.env`,
`docker-compose.yml`, `hf_space`) and set `confidence_threshold` in `api/app/config.py` to `THR`.

In [ ]:
# !python model/publish.py --model "$STAGE2_DIR" --hub_model_id "$HF_MODEL_ID" --dry_run
# !python model/publish.py --model "$STAGE2_DIR" --hub_model_id "$HF_MODEL_ID" --tag v1.1 \
#     --tag_message "Stage 2: PlantVillage + PlantDoc field crops" --commit_message "Stage 2: fine-tune on PlantDoc"

## 8. Publish and tag the release for Phase 5

`model/publish.py` uses the `HF_TOKEN` Colab secret: it checks the token and owner, uploads the final model folder
(weights, `config.json`, `splits.json`, `reports/`, model card; no checkpoints) and tags `v1.0`.
Run the dry run first if you want to see the file list.

Phase 5 pins `HF_MODEL_REVISION=v1.0` so later retraining cannot silently change the API's behaviour.

In [ ]:
!python model/publish.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --dry_run

In [ ]:
!python model/publish.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --tag v1.0

## 9. Inference demo from the Hub (what Phase 5 will do)

In [ ]:
import json, os, random
from PIL import Image as PILImage
from IPython.display import display

import sys
sys.path.insert(0, "model")
from predict import diagnose

m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
threshold = m["recommended_confidence_threshold"]

data_dir = os.environ["DATA_DIR"]
cls = random.choice(sorted(os.listdir(data_dir)))
img_path = os.path.join(data_dir, cls, random.choice(os.listdir(os.path.join(data_dir, cls))))
display(PILImage.open(img_path).resize((224, 224)))
print("True class:", cls)
print(json.dumps(diagnose(img_path, model_id=os.environ["HF_MODEL_ID"], revision="v1.0", threshold=threshold), indent=2))

## 10. Handover to Phase 5

Put these in `.env` of the API service:

```env
HF_MODEL_ID=neghani/plantvillage-mobilenetv2
HF_MODEL_REVISION=v1.0
CONFIDENCE_THRESHOLD=<recommended_confidence_threshold from reports/metrics.json>
```

- `model/predict.py` is the reference implementation (quality gate -> model -> top-k -> not sure).
- Class names in `config.json` (`id2label`) are PlantVillage folder names and match the keys of
  `api/data/knowledge/diseases.json`.
- Full instructions, challenges and the ResNet fallback are in `model/README.md`.